In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving train (1).csv to train (1).csv


# Polars

In [2]:
import polars as pl

### 1. Считывание датасета

In [3]:
dl_titanic = pl.read_csv('train (1).csv')

### 2. Основная информация о датасете:

In [4]:
dl_titanic.describe()

statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


In [5]:
dl_titanic.dtypes

[Int64,
 Int64,
 Int64,
 String,
 String,
 Float64,
 Int64,
 Int64,
 String,
 Float64,
 String,
 String]

In [6]:
dl_titanic.null_count()


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,177,0,0,0,0,687,2


In [7]:
dl_titanic.mean()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null


In [8]:
dl_titanic.shape

(891, 12)

In [9]:
dl_titanic.estimated_size()

85871

### 3. Количество пассажиров каждого класса:

In [10]:
dl_titanic['Pclass'].value_counts().sort('Pclass')

Pclass,count
i64,u32
1,216
2,184
3,491


### 4. Количество выживших мужчин и женщин на корабле:

In [11]:
dl_titanic.filter(pl.col('Survived') == 1).group_by('Sex').len()

Sex,len
str,u32
"""male""",109
"""female""",233


### 5. Пассажиры, возраст которых больше 44 лет:

In [12]:
dl_titanic.filter(pl.col('Age') > 44)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S"""
12,1,1,"""Bonnell, Miss. Elizabeth""","""female""",58.0,0,0,"""113783""",26.55,"""C103""","""S"""
16,1,2,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,0,0,"""248706""",16.0,null,"""S"""
34,0,2,"""Wheadon, Mr. Edward H""","""male""",66.0,0,0,"""C.A. 24579""",10.5,null,"""S"""
53,1,1,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,0,"""PC 17572""",76.7292,"""D33""","""C"""
…,…,…,…,…,…,…,…,…,…,…,…
858,1,1,"""Daly, Mr. Peter Denis ""","""male""",51.0,0,0,"""113055""",26.55,"""E17""","""S"""
863,1,1,"""Swift, Mrs. Frederick Joel (Ma…","""female""",48.0,0,0,"""17466""",25.9292,"""D17""","""S"""
872,1,1,"""Beckwith, Mrs. Richard Leonard…","""female""",47.0,1,1,"""11751""",52.5542,"""D35""","""S"""


# Ускорение работы с pandas

### 1. Считывание датасета

In [13]:
pd_tit = pd.read_csv('train (1).csv')

### 2. Средний возраст пассажиров и его стандартное отклонение (bottleneck)

In [14]:
import bottleneck as bn

In [15]:
bn.nanmean(pd_tit['Age']) ## 177 Nan поэтому используем nanmean

29.69911764705882

In [16]:
bn.nanstd(pd_tit['Age'])

14.516321150817317

### 3. Увеличение столбца Fare на 1.3

In [17]:
pd_tit['Fare_new'] = [row.Fare * 1.3 for row in pd_tit.itertuples()]

In [18]:
pd_tit[['Fare','Fare_new']]

,Fare,Fare_new
0,7.2500,9.42500
1,71.2833,92.66829
2,7.9250,10.30250
3,53.1000,69.03000
4,8.0500,10.46500
...,...,...
886,13.0000,16.90000
887,30.0000,39.00000
888,23.4500,30.48500
889,30.0000,39.00000


# Оптимизация типов pandas

### 1. Считывание датасета

In [19]:
uploaded = files.upload()

Saving Housing.csv to Housing.csv


In [20]:
pd_housing = pd.read_csv('Housing.csv')

In [21]:
pd_housing.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [22]:
pd_housing.dtypes

,0
price,int64
area,int64
bedrooms,int64
bathrooms,int64
stories,int64
mainroad,object
guestroom,object
basement,object
hotwaterheating,object
airconditioning,object


In [23]:
pd_housing.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [24]:
pd_housing.columns[pd_housing.isna().any()]

Index([], dtype='object')

### 2-3. Оптимизация типов данных

In [25]:
old_size = pd_housing.memory_usage(deep=True).sum()
old_size

np.int64(227244)

Для каждого столбца выбран минимально достаточный тип: `price` - `uint32` (max 13.3M < 4.3B), `area` - `int16` (max = 16200 < 32767), `bedrooms/bathrooms/stories/parking` - `int8` (max = 6 < 127), булевые столбцы - `bool`, `furnishingstatus` - `category` (3 уникальных значения).

In [26]:
from numpy import uint32, int16, int8
pd_housing['price'] = pd_housing['price'].astype(uint32)
pd_housing['area'] = pd_housing['area'].astype(int16)
pd_housing['bedrooms'] = pd_housing['bedrooms'].astype(int8)
pd_housing['bathrooms'] = pd_housing['bathrooms'].astype(int8)
pd_housing['stories'] = pd_housing['stories'].astype(int8)
pd_housing['mainroad'] = pd_housing['mainroad'].astype(bool)
pd_housing['guestroom'] = pd_housing['guestroom'].astype(bool)
pd_housing['basement'] = pd_housing['basement'].astype(bool)
pd_housing['hotwaterheating'] = pd_housing['hotwaterheating'].astype(bool)
pd_housing['airconditioning'] = pd_housing['airconditioning'].astype(bool)
pd_housing['prefarea'] = pd_housing['prefarea'].astype(bool)
pd_housing['parking'] = pd_housing['parking'].astype(int8)
pd_housing['furnishingstatus'] = pd_housing['furnishingstatus'].astype('category')


In [27]:
new_size = pd_housing.memory_usage(deep=True).sum()

In [29]:
print(f"optimized dataset takes {old_size/new_size:.1f} times less space")

optimized dataset takes 23.5 times less space
